# Receipt OCR and export summary

## Imports

In [4]:
!pip install easyocr > /dev/null

In [31]:
import easyocr
import re
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np

## Consts

In [6]:
RAW_DIR = './img/'
PRE_PROCESSED_DIR = './preprocessed/'

In [28]:
# @title Image Pre-processor

def preprocess_image(image_path):
    print(f'Preprocessing <{image_path}>...', end=' ')
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 145, 255, cv2.THRESH_TOZERO)
    denoised = cv2.medianBlur(binary, 3)  # Remove small noise
    resized = cv2.resize(denoised, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
    print('DONE')
    return resized

In [67]:
# @title Draw Image

def draw_image(imgs, title="Figure", gray=False):
    num_images = len(imgs)

    # Calculate rows and columns for subplots
    cols = min(3, num_images)  # Limit to 3 columns by default
    rows = int(np.ceil(num_images / cols))

    # Create figure and subplots
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3 + 3, rows * 3 + 3))
    fig.suptitle(title, fontsize=16)

    # Ensure axes is always iterable
    axes = np.array(axes).reshape(-1)

    for i, ax in enumerate(axes):
        if i < num_images:
            img = imgs[i]
            if gray:
                ax.imshow(img, cmap='gray')
            else:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(f"Image {i + 1}")
        else:
            ax.axis('off')  # Hide unused subplots
        ax.axis('off')  # Turn off axes for all subplots

    # Adjust layout for better spacing
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [45]:
# @title Draw Detections

def draw_detections(image_path, results):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Draw bounding boxes and text
    for detection in results:
        top_left = tuple(map(int, detection[0][0]))
        bottom_right = tuple(map(int, detection[0][2]))
        text = detection[1]
        confidence = detection[2]

        # Rectangle
        cv2.rectangle(img, top_left, bottom_right, (0, 255, 0), 2)
        # Text
        cv2.putText(img, f"{text} ({confidence:.2f})", (top_left[0], top_left[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    draw_image([img], title=f'{image_path}', gray=False)

In [29]:
for image in os.listdir(RAW_DIR):
    image_path = os.path.join(RAW_DIR, image)
    preprocessed_path = os.path.join(PRE_PROCESSED_DIR, image)
    preprocessed = preprocess_image(image_path)
    cv2.imwrite(preprocessed_path, preprocessed)

Preprocessing <./img/receipt.jpg>... DONE


In [68]:
# Checking one of the preprocessed images

img_path = os.listdir(RAW_DIR)[0]
pre = cv2.imread(os.path.join(RAW_DIR, img_path))
post = cv2.imread(os.path.join(PRE_PROCESSED_DIR,img_path))

draw_image([pre, post], title='Pre vs Post')

<Figure size 900x600 with 2 Axes>

In [15]:
total_results = {}

for image in os.listdir(PRE_PROCESSED_DIR):
    print(f'Reading <{image}>', end='\n\n')

    image_path = os.path.join(PRE_PROCESSED_DIR, image)

    reader = easyocr.Reader(['es'])
    results = reader.readtext(image_path)
    total_results[image_path] = results

Reading <receipt.jpg>



In [69]:
img_path, res = list(total_results.items())[0]
draw_detections(img_path, res)

<Figure size 600x600 with 1 Axes>

In [ ]:
reader = easyocr.Reader(['es'])  # Specify language(s)
results = reader.readtext(image_path)

In [ ]:
# def extract_receipt_details(image_path):
#     reader = easyocr.Reader(['es'])  # Specify language(s)
#     results = reader.readtext(image_path)

#     # Combine detected text into a single string
#     text = " ".join([res[1] for res in results])

#     # Extract fields using regex
#     date = re.search(r'\b\d{2}/\d{2}/\d{4}\b', text)
#     total = re.search(r'\bTotal:\s*\$\d+\.\d{2}\b', text, re.IGNORECASE)

#     return {
#         "text": text,
#         "date": date.group(0) if date else None,
#         "total": total.group(0) if total else None
#     }

# details = extract_receipt_details('receipt.jpg')
# print(details)


In [ ]:
# import easyocr
# import cv2
# import matplotlib.pyplot as plt

# def visualize_text_detection(image_path):
#     # Initialize EasyOCR reader
#     reader = easyocr.Reader(['en'])

#     # Perform OCR and get results
#     results = reader.readtext(image_path)

#     # Load the image using OpenCV
#     img = cv2.imread(image_path)
#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB for correct color display

#     # Draw bounding boxes and text
#     for detection in results:
#         top_left = tuple(map(int, detection[0][0]))
#         bottom_right = tuple(map(int, detection[0][2]))
#         text = detection[1]
#         confidence = detection[2]

#         # Draw the rectangle
#         cv2.rectangle(img, top_left, bottom_right, (0, 255, 0), 2)
#         # Put the text
#         cv2.putText(img, f"{text} ({confidence:.2f})", (top_left[0], top_left[1] - 10),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

#     # Display the image with matplotlib
#     plt.figure(figsize=(10, 10))
#     plt.imshow(img)
#     plt.axis('off')
#     plt.show()

# # Test the function
# visualize_text_detection('receipt.jpg')